In [1]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

WARD_URL = "https://integrity.ng/index.php/wards/browse"
PU_URL = "https://integrity.ng/index.php/units/browse/"
MAX_PAGES = 200


def init_driver():
    options = Options()
    options.add_argument("--headless")  # comment out to view browser
    options.add_argument("--window-size=1920,1080")
    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=options)


def fetch_lgas_and_wards(state_name, max_pages=MAX_PAGES):
    print(f"🔍 Fetching wards for {state_name}...")
    driver = init_driver()
    wards = []
    seen = set()
    page = 0

    while page < max_pages:
        driver.get(f"{WARD_URL}?page={page}")
        try:
            table = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, "table")))
            rows = table.find_elements(By.TAG_NAME, "tr")[1:]
            if not rows:
                break

            for row in rows:
                cols = row.find_elements(By.TAG_NAME, "td")
                if len(cols) >= 4:
                    state = cols[1].text.strip()
                    lga = cols[2].text.strip()
                    ward = cols[3].text.strip()

                    if state.lower() == state_name.lower():
                        key = f"{lga}-{ward}"
                        if key not in seen:
                            wards.append({"LGA": lga, "Ward": ward})
                            seen.add(key)

            print(f"✅ Page {page} collected, Total: {len(wards)}")
            page += 1
            time.sleep(1)

        except Exception as e:
            print(f"⚠️ Error on page {page}: {e}")
            break

    driver.quit()
    return wards


def fetch_polling_units(state_name, wards, max_pages=MAX_PAGES):
    print(f"\n🔍 Fetching polling units for {state_name}...")
    driver = init_driver()
    pus = []
    seen = set()
    page = 0

    while page < max_pages:
        driver.get(f"{PU_URL}?page={page}")
        try:
            table = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, "table")))
            rows = table.find_elements(By.TAG_NAME, "tr")[1:]
            if not rows:
                break

            for row in rows:
                cols = row.find_elements(By.TAG_NAME, "td")
                if len(cols) >= 5:
                    state = cols[1].text.strip()
                    lga = cols[2].text.strip()
                    ward = cols[3].text.strip()
                    pu = cols[4].text.strip()

                    if state.lower() == state_name.lower():
                        key = f"{ward}-{pu}"
                        if key not in seen:
                            pus.append({"Ward": ward, "Polling Unit": pu})
                            seen.add(key)

            print(f"✅ Page {page} collected, Total: {len(pus)}")
            page += 1
            time.sleep(1)

        except Exception as e:
            print(f"⚠️ Error on PU page {page}: {e}")
            break

    driver.quit()
    return pus


def save_to_excel(state_name, wards, pus):
    df_lga = pd.DataFrame(sorted(set([w["LGA"] for w in wards])), columns=["LGA"])
    df_wards = pd.DataFrame(wards).drop_duplicates()
    df_pus = pd.DataFrame(pus).drop_duplicates()

    file_name = f"{state_name}_LGAs_Wards_PUs.xlsx"
    with pd.ExcelWriter(file_name, engine="openpyxl") as writer:
        df_lga.to_excel(writer, sheet_name="LGAs", index=False)
        df_wards.to_excel(writer, sheet_name="LGA_Wards", index=False)
        df_pus.to_excel(writer, sheet_name="Wards_PUs", index=False)

    print(f"\n✅ Saved: {file_name} ({len(df_lga)} LGAs, {len(df_wards)} wards, {len(df_pus)} polling units)\n")


def run_for_state(state_name):
    print(f"\n🚀 Starting for {state_name}")
    wards = fetch_lgas_and_wards(state_name)
    if not wards:
        print(f"❌ No wards found for {state_name}.")
        return

    pus = fetch_polling_units(state_name, wards)
    save_to_excel(state_name, wards, pus)


if __name__ == "__main__":
    state_input = input("Enter state name (e.g., Katsina): ").strip()
    run_for_state(state_input)



🚀 Starting for Katsina
🔍 Fetching wards for Katsina...
✅ Page 0 collected, Total: 0
✅ Page 1 collected, Total: 0
✅ Page 2 collected, Total: 0
✅ Page 3 collected, Total: 0
✅ Page 4 collected, Total: 0
✅ Page 5 collected, Total: 0
✅ Page 6 collected, Total: 0
✅ Page 7 collected, Total: 0
✅ Page 8 collected, Total: 0
✅ Page 9 collected, Total: 0
✅ Page 10 collected, Total: 0
✅ Page 11 collected, Total: 0
✅ Page 12 collected, Total: 0
✅ Page 13 collected, Total: 0
✅ Page 14 collected, Total: 0
✅ Page 15 collected, Total: 0
✅ Page 16 collected, Total: 0
✅ Page 17 collected, Total: 0
✅ Page 18 collected, Total: 0
✅ Page 19 collected, Total: 0
✅ Page 20 collected, Total: 0
✅ Page 21 collected, Total: 0
✅ Page 22 collected, Total: 0
✅ Page 23 collected, Total: 0
✅ Page 24 collected, Total: 0
✅ Page 25 collected, Total: 0
✅ Page 26 collected, Total: 0
✅ Page 27 collected, Total: 0
✅ Page 28 collected, Total: 0
✅ Page 29 collected, Total: 0
✅ Page 30 collected, Total: 0
✅ Page 31 collected, Tot